In [24]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix

In [25]:
batch_size = 16
image_size = (128, 128)

In [26]:
# 1. Data Generators (FIXED)
# ----------------------------
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    "split_data/train",
    target_size=(224, 224),
    batch_size=16,
    class_mode='categorical'
)

Found 13726 images belonging to 5 classes.


In [27]:
val_data = val_datagen.flow_from_directory(
    "split_data/val",
    target_size=(224, 224),
    batch_size=16,
    class_mode='categorical',
    shuffle=False
)


Found 2722 images belonging to 5 classes.


In [28]:
# 2. Base Model (ResNet50)
# ----------------------------
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze layers
for layer in base_model.layers:
    layer.trainable = False


In [29]:
# 3. Custom Head
# ----------------------------
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(train_data.num_classes, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

In [30]:
# ----------------------------
# 4. Compile
# ----------------------------
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ----------------------------
# 5. Callbacks
# ----------------------------
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "best_resnet50.h5",
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

In [31]:
# ----------------------------
# 6. Train (Phase 1)
# ----------------------------
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=[early_stop, checkpoint]
)

# ----------------------------
# 7. Fine-Tuning (IMPORTANT)
# ----------------------------
for layer in base_model.layers[-30:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


Epoch 1/20
858/858 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4352 - loss: 1.3404
Epoch 1: val_accuracy improved from None to 0.56943, saving model to best_resnet50.h5


858/858 ━━━━━━━━━━━━━━━━━━━━ 1272s 1s/step - accuracy: 0.4967 - loss: 1.2281 - val_accuracy: 0.5694 - val_loss: 1.0415
Epoch 2/20
858/858 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5688 - loss: 1.0823
Epoch 2: val_accuracy improved from 0.56943 to 0.64144, saving model to best_resnet50.h5


858/858 ━━━━━━━━━━━━━━━━━━━━ 1278s 1s/step - accuracy: 0.5857 - loss: 1.0505 - val_accuracy: 0.6414 - val_loss: 0.8857
Epoch 3/20
858/858 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6209 - loss: 0.9735
Epoch 3: val_accuracy improved from 0.64144 to 0.66091, saving model to best_resnet50.h5


858/858 ━━━━━━━━━━━━━━━━━━━━ 1264s 1s/step - accuracy: 0.6214 - loss: 0.9679 - val_accuracy: 0.6609 - val_loss: 0.8456
Epoch 4/20
858/858 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6380 - loss: 0.9297
Epoch 4: val_accuracy improved from 0.66091 to 0.69140, saving model to best_resnet50.h5


858/858 ━━━━━━━━━━━━━━━━━━━━ 1250s 1s/step - accuracy: 0.6446 - loss: 0.9140 - val_accuracy: 0.6914 - val_loss: 0.8076
Epoch 5/20
858/858 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6683 - loss: 0.8876
Epoch 5: val_accuracy did not improve from 0.69140
858/858 ━━━━━━━━━━━━━━━━━━━━ 1269s 1s/step - accuracy: 0.6695 - loss: 0.8739 - val_accuracy: 0.6785 - val_loss: 0.7920
Epoch 6/20
858/858 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6693 - loss: 0.8533
Epoch 6: val_accuracy improved from 0.69140 to 0.70279, saving model to best_resnet50.h5


...

858/858 ━━━━━━━━━━━━━━━━━━━━ 1267s 1s/step - accuracy: 0.7267 - loss: 0.7173 - val_accuracy: 0.7513 - val_loss: 0.6599
Epoch 14/20
858/858 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7303 - loss: 0.7106
Epoch 14: val_accuracy did not improve from 0.75129
858/858 ━━━━━━━━━━━━━━━━━━━━ 1232s 1s/step - accuracy: 0.7326 - loss: 0.7032 - val_accuracy: 0.7491 - val_loss: 0.6518
Epoch 15/20
858/858 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7275 - loss: 0.7101
Epoch 15: val_accuracy improved from 0.75129 to 0.75533, saving model to best_resnet50.h5


858/858 ━━━━━━━━━━━━━━━━━━━━ 1256s 1s/step - accuracy: 0.7336 - loss: 0.6960 - val_accuracy: 0.7553 - val_loss: 0.6381
Epoch 16/20
858/858 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7387 - loss: 0.6816
Epoch 16: val_accuracy improved from 0.75533 to 0.75974, saving model to best_resnet50.h5


858/858 ━━━━━━━━━━━━━━━━━━━━ 1264s 1s/step - accuracy: 0.7349 - loss: 0.6902 - val_accuracy: 0.7597 - val_loss: 0.6191
Epoch 17/20
858/858 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7455 - loss: 0.6582
Epoch 17: val_accuracy improved from 0.75974 to 0.76561, saving model to best_resnet50.h5


858/858 ━━━━━━━━━━━━━━━━━━━━ 1249s 1s/step - accuracy: 0.7385 - loss: 0.6717 - val_accuracy: 0.7656 - val_loss: 0.6078
Epoch 18/20
858/858 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7467 - loss: 0.6637
Epoch 18: val_accuracy did not improve from 0.76561
858/858 ━━━━━━━━━━━━━━━━━━━━ 1247s 1s/step - accuracy: 0.7411 - loss: 0.6726 - val_accuracy: 0.7634 - val_loss: 0.6338
Epoch 19/20
858/858 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7451 - loss: 0.6569
Epoch 19: val_accuracy improved from 0.76561 to 0.77847, saving model to best_resnet50.h5


858/858 ━━━━━━━━━━━━━━━━━━━━ 1243s 1s/step - accuracy: 0.7481 - loss: 0.6599 - val_accuracy: 0.7785 - val_loss: 0.6007
Epoch 20/20
858/858 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7453 - loss: 0.6487
Epoch 20: val_accuracy did not improve from 0.77847
858/858 ━━━━━━━━━━━━━━━━━━━━ 1249s 1s/step - accuracy: 0.7474 - loss: 0.6504 - val_accuracy: 0.7752 - val_loss: 0.5900


In [32]:
# ----------------------------
# 8. Save Final Model
# ----------------------------
model.save("ResNet50_final.h5")

In [34]:
# Reset generator (IMPORTANT)
val_data.reset()

# Predictions
predictions = model.predict(val_data)

# Convert to class index
y_pred = np.argmax(predictions, axis=1)

# True labels
y_true = val_data.classes

# Class names
class_names = list(val_data.class_indices.keys())

171/171 ━━━━━━━━━━━━━━━━━━━━ 199s 1s/step


In [35]:
# ----------------------------
# 10. Evaluation
# ----------------------------
print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

print("\nConfusion Matrix:\n")
cm = confusion_matrix(y_true, y_pred)
print(cm)


Classification Report:

              precision    recall  f1-score   support

       Boron       0.87      0.72      0.79       366
     Healthy       0.91      0.89      0.90       463
      Kalium       0.68      0.80      0.73       777
          Mg       0.73      0.78      0.76       734
    nitrogen       0.91      0.63      0.74       382

    accuracy                           0.78      2722
   macro avg       0.82      0.76      0.78      2722
weighted avg       0.79      0.78      0.78      2722


Confusion Matrix:

[[263   0  84  18   1]
 [  1 413  29  11   9]
 [ 27  16 622 106   6]
 [ 12  21 120 573   8]
 [  0   6  63  74 239]]


In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image

# Load model
model = tf.keras.models.load_model("best_resnet50.h5")  # or ResNet50_final.h5

# Class labels (IMPORTANT)
class_names = list(val_data.class_indices.keys())

# Load image
img_path = "test.jpg"   # 🔁 change this
img = image.load_img(img_path, target_size=(224, 224))

# Preprocess
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Predict
pred = model.predict(img_array)
pred_class = np.argmax(pred, axis=1)[0]
confidence = np.max(pred)

# Output
print("Predicted:", class_names[pred_class])
print("Confidence:", confidence)

In [ ]:
import json

with open("labels.json", "w") as f:
    json.dump(train_data.class_indices, f)

In [ ]:
import json

with open("labels.json", "r") as f:
    class_indices = json.load(f)

class_names = list(class_indices.keys())

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(img)
plt.title(f"{class_names[pred_class]} ({confidence:.2f})")
plt.axis('off')
plt.show()